# Bronze Layer - pump.fun Event Ingestion (Delta Live Tables)

Delta Live Tables pipeline source. This is **not** a regular job notebook —
it must be run as the source code of a DLT pipeline (see
`orchestration/bundle/databricks.yml`, resource `pumpfun-bronze-dlt`), which
provides the `dlt` module and manages the streaming execution.

The `ingestion/pumpfun` producer batches events, serializes each batch as
JSONL, compresses it with zstd, and base64-encodes the result into a single
JSON envelope per landed file:

```json
{"_source": "pumpapi", "_ingested_at": "...", "event_count": 5000,
 "codec": "zstd", "uncompressed_bytes": 6673900, "data_b64": "..."}
```

This pipeline reverses that: base64-decode -> zstd-decompress -> split back
into JSONL -> parse each line's `event` into `bronze.pumpfun_events`, one
row per pump.fun event — the same final Bronze shape produced by the
now-outdated `outdated__NB_ingest_pumpfun_to_bronze.ipynb`, so
`processing/silver/NB_process_pumpfun_silver.ipynb` needs no changes.

In [ ]:
%pip install zstandard

In [ ]:
import base64

import dlt
import zstandard as zstd
from pyspark.sql.functions import col, current_timestamp, explode, get_json_object, split, udf
from pyspark.sql.types import LongType, StringType, StructField, StructType

LANDING_PATH = spark.conf.get("pumpfun.landing_path", "/Volumes/workspace/default/mnt/pumpapi")

# Fixed schema for the envelope files written by ingestion/pumpfun/app/writer.py.
# No Auto Loader schema inference/evolution needed -- this shape is controlled by us.
_ENVELOPE_SCHEMA = StructType([
    StructField("_source",            StringType(), True),
    StructField("_ingested_at",       StringType(), True),
    StructField("event_count",        LongType(),   True),
    StructField("codec",              StringType(), True),
    StructField("uncompressed_bytes", LongType(),   True),
    StructField("data_b64",           StringType(), True),
])

In [ ]:
_decompressors = {
    "zstd": zstd.ZstdDecompressor(),
}


def _decode_envelope(data_b64: str, codec: str) -> str | None:
    """base64-decode + decompress one envelope's payload back into JSONL text."""
    if data_b64 is None or codec is None:
        return None

    decompressor = _decompressors.get(codec)
    if decompressor is None:
        # Unsupported codec: surface as a null jsonl_text rather than failing the
        # pipeline -- these rows are easy to spot and backfill once handled.
        return None

    compressed = base64.b64decode(data_b64)
    return decompressor.decompress(compressed).decode("utf-8")


decode_envelope = udf(_decode_envelope, StringType())

In [ ]:
@dlt.table(
    name="pumpfun_events",
    comment="Raw pump.fun events, decoded from the zstd+base64 batch envelopes landed by ingestion/pumpfun.",
    table_properties={"quality": "bronze"},
)
def pumpfun_events():
    envelopes = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(_ENVELOPE_SCHEMA)
        .load(LANDING_PATH)
    )

    decoded = (
        envelopes
        .withColumn("jsonl_text", decode_envelope(col("data_b64"), col("codec")))
        .filter(col("jsonl_text").isNotNull())
    )

    lines = (
        decoded
        .withColumn("line", explode(split(col("jsonl_text"), "\n")))
        .filter(col("line") != "")
    )

    return (
        lines
        .withColumn("source",             get_json_object(col("line"), "$._source"))
        .withColumn("ingested_at",        get_json_object(col("line"), "$._ingested_at").cast("timestamp"))
        .withColumn("event_json",         get_json_object(col("line"), "$.event"))
        .withColumn("action",             get_json_object(col("event_json"), "$.action"))
        .withColumn("mint",               get_json_object(col("event_json"), "$.mint"))
        .withColumn("signature",          get_json_object(col("event_json"), "$.signature"))
        .withColumn("pool_id",            get_json_object(col("event_json"), "$.poolId"))
        .withColumnRenamed("line",        "raw_line")
        .withColumn("bronze_ingested_at", current_timestamp())
        .select(
            "source", "ingested_at", "action", "mint", "signature", "pool_id",
            "event_json", "raw_line", "bronze_ingested_at",
        )
    )